# C3 可选实验：横向比较索引增强字段

本页把 CCH（可靠章节标题）、Document Augmentation（模型生成问题）和 Contextual Retrieval（模型生成背景）放进同一个小型比较。它是索引阶段的可选横向实验；[专题页](给片段补充章节标题.ipynb)、[Document Augmentation 的说明](给片段补充检索说法.ipynb)和[Contextual Retrieval 的说明](给片段补充所属上下文.ipynb)仍然分别用于学习单一机制。

所有方法使用同一批来自《南瓜书》的 3 个问题、同一组 9 页候选语料、同一个 BM25 检索器、`top_k=3` 和同一个检索字段字符预算。原文方法是基线；增强方法只改变索引字段，命中后按稳定 `chunk_id` 映回 `answer_text`（原始 PDF 文字）。回答映回只核查来源，不组装或比较回答上下文；本实验不调用回答模型，因此表格中的排名是检索结果，不是答案质量。

GLM 生成问题和背景各自保存在独立字段。原始 API 返回、结构校验记录、失败策略和每道题相对基线的改善/不变/退化都会写入 Notebook 输出；如果真实调用或 JSON 校验失败，代码会记录并直接报错，绝不使用 fallback 或伪造文字。

In [1]:
from __future__ import annotations

import re
import sys
from pathlib import Path


def find_tutorial_root(start: Path) -> Path:
    for folder in (start.resolve(), *start.resolve().parents):
        if (folder / "data" / "dataset" / "manifest.json").is_file() and (
            folder / "common" / "eval_utils.py"
        ).is_file():
            return folder
    raise FileNotFoundError("找不到 C7 教程根目录，请从 C7 目录或本 Notebook 运行。")


TUTORIAL_ROOT = find_tutorial_root(Path.cwd())
if str(TUTORIAL_ROOT) not in sys.path:
    sys.path.insert(0, str(TUTORIAL_ROOT))

from common.dataset import chapter_title


from common.eval_utils import (
    build_bm25_chunk_search,
    emit_tutorial_audit,
    load_pdf_pages,
)
from common.nontraining_utils import (
    PROJECT_ENV_RELATIVE_PATH,
    load_annotation,
    load_query_only,
    llm_call,
    parse_json_object,
)


QUESTION_IDS = [
    "model_selection_with_intro_scope",
    "model_evaluation_followup",
    "ensemble_learning_definition",
]
queries = load_query_only(QUESTION_IDS)
query_by_id = {item["id"]: item["query"] for item in queries}

# 这是预先固定的小候选语料：所有方法共享它，不按某个方法的结果重新选语料。
CANDIDATE_PAGES = (2, 14, 15, 16, 17, 18, 19, 88, 101)
TOP_K = 3
SEARCH_CHAR_BUDGET = 400

pages = load_pdf_pages()
page_by_number = {int(row["page"]): row for row in pages}
missing_pages = [page for page in CANDIDATE_PAGES if page not in page_by_number]
if missing_pages:
    raise ValueError(f"候选语料缺少 PDF 页：{missing_pages}")

candidate_rows = []
for page in CANDIDATE_PAGES:
    original = str(page_by_number[page]["text"]).strip()
    if not original:
        raise ValueError(f"候选页没有原文：{page}")
    candidate_rows.append(
        {
            "chunk_id": f"c3-page-{page}",
            "page": page,
            "pages": [page],
            "original_text": original,
            "answer_text": original,
            "chapter_title": chapter_title(page),
        }
    )

if len({row["chunk_id"] for row in candidate_rows}) != len(candidate_rows):
    raise ValueError("候选语料 chunk_id 必须稳定且唯一")
if any(row["answer_text"] != row["original_text"] for row in candidate_rows):
    raise AssertionError("answer_text 必须在生成增强字段前严格等于原始片段")

print("模型：", "glm-4-flash")
print("真实配置：", PROJECT_ENV_RELATIVE_PATH)
print("问题数：", len(queries), "；候选页数：", len(candidate_rows))
print("候选页：", list(CANDIDATE_PAGES))
print("统一配置：top_k=", TOP_K, "；检索字段字符预算=", SEARCH_CHAR_BUDGET, "；回答映回只核查来源，不组装/比较回答上下文")

模型： glm-4-flash
真实配置： .env
问题数： 3 ；候选页数： 9
候选页： [2, 14, 15, 16, 17, 18, 19, 88, 101]
统一配置：top_k= 3 ；检索字段字符预算= 400；回答映回只核查来源，不组装/比较回答上下文


## 生成两个增强字段，并保留原始产物

每个候选页只做一次真实 `glm-4-flash` 调用，同时请求两个互不混淆的字段：`questions` 用于 Document Augmentation，`contextual_background` 用于 Contextual Retrieval。可靠章节标题不由模型生成，而是来自已核对的 PDF 页码范围。模型看不到问题集的 `expected_pages` 或参考答案；这些标注只在所有方法检索完成后才读取。

In [2]:
def generation_prompt(row: dict) -> str:
    return (
        "你在为《南瓜书》建立检索索引。只根据下面给出的原文生成 JSON，不要使用外部知识。\n"
        "原文所属的可靠章节标题：" + row["chapter_title"] + "\n"
        "当前页原文：\n<original>\n" + row["original_text"] + "\n</original>\n\n"
        "请严格返回一个 JSON 对象，不要 Markdown 代码围栏，字段必须是：\n"
        "{\"questions\":[\"问题1\",\"问题2\"],"
        "\"contextual_background\":\"一句背景\"}.\n"
        "questions 是 1 到 2 个能由原文回答的读者问题，每个不超过 96 个字，不能直接写答案；"
        "contextual_background 是一句不超过 160 个字的索引背景，只说明片段在章节中的位置和作用，"
        "不能回答某个用户问题，不能编造原文没有的事实、页码或数字。"
    )


def validate_generation(raw: str) -> dict:
    raw_text = str(raw or "").strip()
    if not raw_text:
        raise ValueError("模型返回空文字")
    payload = parse_json_object(raw_text)
    questions = payload.get("questions")
    context = payload.get("contextual_background")
    if not isinstance(questions, list) or not 1 <= len(questions) <= 2:
        raise ValueError("questions 必须是 1 到 2 项的列表")
    if any(not isinstance(item, str) or not item.strip() for item in questions):
        raise ValueError("questions 中每一项都必须是非空文字")
    cleaned_questions = [item.strip() for item in questions]
    if any(len(item) > 96 or len(item.splitlines()) != 1 for item in cleaned_questions):
        raise ValueError("questions 中每一项必须是不超过 96 字的单行文字")
    if not isinstance(context, str) or not context.strip():
        raise ValueError("contextual_background 必须是非空文字")
    context = context.strip()
    if len(context) > 160 or len(context.splitlines()) != 1:
        raise ValueError("contextual_background 必须是不超过 160 字的单行文字")
    return {
        "questions": cleaned_questions,
        "contextual_background": context,
        "validation_notes": [
            "JSON 对象结构通过",
            "questions 为 1–2 条非空单行文字且长度合规",
            "contextual_background 为非空单行文字且长度合规",
        ],
    }


generation_records = {}
generation_failures = []
for row in candidate_rows:
    chunk_id = row["chunk_id"]
    raw_output = ""
    try:
        raw_output = llm_call(generation_prompt(row), max_tokens=320)
        parsed = validate_generation(raw_output)
    except Exception as error:
        # 原始返回（若有）和错误都留下；没有任何占位文字或静默回退。
        record = {
            "chunk_id": chunk_id,
            "raw_model_output": raw_output,
            "status": "failed",
            "error": str(error)[:400],
            "validation_notes": ["结构校验失败；本次索引不能继续"],
        }
        generation_records[chunk_id] = record
        generation_failures.append(record)
        print("生成失败：", chunk_id, record["error"])
        continue
    generation_records[chunk_id] = {
        "chunk_id": chunk_id,
        "raw_model_output": raw_output,
        "parsed_payload": parsed,
        "status": "validated",
        "validation_notes": parsed["validation_notes"],
    }

print("GLM 生成任务数：", len(candidate_rows))
print("结构校验通过：", sum(record["status"] == "validated" for record in generation_records.values()))
print("结构校验失败：", len(generation_failures), "（失败时不回退，直接终止索引构建）")
if generation_failures:
    emit_tutorial_audit({
        "case_id": "c3_index_enhancement_generation_contract",
        "method": "C3 索引增强横向比较",
        "role": "generation",
        "generation_records": generation_records,
        "provenance": {"status": "failed_no_fallback", "model": "glm-4-flash"},
    })
    raise RuntimeError("至少一个真实 GLM 输出未通过结构校验；没有使用 fallback，已停止。")

row_by_id = {}
for row in candidate_rows:
    generated = generation_records[row["chunk_id"]]["parsed_payload"]
    enriched = {
        **row,
        "generated_questions": generated["questions"],
        "contextual_background": generated["contextual_background"],
        "raw_model_output": generation_records[row["chunk_id"]]["raw_model_output"],
        "validation_status": generation_records[row["chunk_id"]]["status"],
        "validation_notes": generation_records[row["chunk_id"]]["validation_notes"],
    }
    row_by_id[row["chunk_id"]] = enriched

METHODS = {
    "原文": "original_text",
    "可靠章节标题（CCH）": "chapter_title",
    "GLM生成问题（Document Augmentation）": "generated_questions",
    "GLM背景（Contextual Retrieval）": "contextual_background",
}


def compose_search_text(row: dict, method: str) -> str:
    if method == "原文":
        text = row["original_text"]
    elif method == "可靠章节标题（CCH）":
        text = f"{row['chapter_title']}\n{row['original_text']}"
    elif method == "GLM生成问题（Document Augmentation）":
        text = f"{'；'.join(row['generated_questions'])}\n{row['original_text']}"
    elif method == "GLM背景（Contextual Retrieval）":
        text = f"{row['contextual_background']}\n{row['original_text']}"
    else:
        raise KeyError(f"未知索引方法：{method}")
    # 对每个方法使用同一个预算；answer_text 永远不在此处截断或改写。
    return text.strip()[:SEARCH_CHAR_BUDGET]

index_rows_by_method = {}
for method in METHODS:
    index_rows = []
    for chunk_id, row in row_by_id.items():
        search_text = compose_search_text(row, method)
        row["search_text_by_method"] = {
            **row.get("search_text_by_method", {}),
            method: search_text,
        }
        index_rows.append(
            {
                "chunk_id": chunk_id,
                "pages": row["pages"],
                "text": search_text,
            }
        )
    if len(index_rows) != len(candidate_rows):
        raise AssertionError(f"{method} 没有使用同一候选语料")
    index_rows_by_method[method] = index_rows

searchers = {
    method: build_bm25_chunk_search(index_rows)
    for method, index_rows in index_rows_by_method.items()
}
print("比较方法：", list(METHODS))
print("每种方法的索引条数：", {method: len(rows) for method, rows in index_rows_by_method.items()})
print("每种方法的真实字段：", METHODS)

GLM 生成任务数： 9
结构校验通过： 9
结构校验失败： 0 （失败时不回退，直接终止索引构建）
比较方法： ['原文', '可靠章节标题（CCH）', 'GLM生成问题（Document Augmentation）', 'GLM背景（Contextual Retrieval）']
每种方法的索引条数： {'原文': 9, '可靠章节标题（CCH）': 9, 'GLM生成问题（Document Augmentation）': 9, 'GLM背景（Contextual Retrieval）': 9}
每种方法的真实字段： {'原文': 'original_text', '可靠章节标题（CCH）': 'chapter_title', 'GLM生成问题（Document Augmentation）': 'generated_questions', 'GLM背景（Contextual Retrieval）': 'contextual_background'}


## 同题、同语料、同预算比较

先让四种索引各自完成检索，再读取评测标注计算必要页的首条排名和覆盖率。每个增强方法都和同一问题的“原文”基线比较：`improved` 表示排名变小或覆盖率变高，`degraded` 表示排名变大或覆盖率变低，`unchanged` 表示两项都没有变化，二者同时发生时记为 `tradeoff`。因此结果可以诚实地显示无收益或退化。

In [3]:
def metrics_for(hits, expected_pages):
    expected = {int(page) for page in expected_pages}
    pages = [int(page) for hit in hits for page in hit.pages]
    found = sorted(expected.intersection(pages))
    first_rank = next(
        (rank for rank, hit in enumerate(hits, 1) if expected.intersection(hit.pages)),
        None,
    )
    return {
        "pages": pages,
        "first_required_rank": first_rank,
        "required_pages_found": found,
        "required_page_coverage": len(found) / len(expected) if expected else 0.0,
    }


def rank_value(rank):
    return TOP_K + 1 if rank is None else int(rank)


def outcome_vs_baseline(before, after):
    coverage_before = float(before["required_page_coverage"])
    coverage_after = float(after["required_page_coverage"])
    rank_before = rank_value(before["first_required_rank"])
    rank_after = rank_value(after["first_required_rank"])
    improved = coverage_after > coverage_before or rank_after < rank_before
    degraded = coverage_after < coverage_before or rank_after > rank_before
    if improved and degraded:
        return "tradeoff"
    if improved:
        return "improved"
    if degraded:
        return "degraded"
    return "unchanged"


def answer_evidence(hits):
    mapped = []
    for hit in hits:
        if hit.chunk_id not in row_by_id:
            raise KeyError(f"检索结果的 chunk_id 无法映回原文：{hit.chunk_id}")
        source = row_by_id[hit.chunk_id]
        mapped.append(
            {
                "chunk_id": hit.chunk_id,
                "pages": list(hit.pages),
                "answer_text": source["answer_text"],
            }
        )
    return mapped


# 先完成所有方法的检索；评测页码在这个循环之后才读取。
retrieval_results = {
    case_id: {
        method: searchers[method](query_by_id[case_id], top_k=TOP_K)
        for method in METHODS
    }
    for case_id in QUESTION_IDS
}
annotations = {case_id: load_annotation(case_id) for case_id in QUESTION_IDS}

benchmark = {}
for case_id in QUESTION_IDS:
    baseline_metrics = metrics_for(
        retrieval_results[case_id]["原文"],
        annotations[case_id]["expected_pages"],
    )
    benchmark[case_id] = {}
    for method in METHODS:
        hits = retrieval_results[case_id][method]
        if len(hits) != TOP_K:
            raise AssertionError(f"{method} 没有使用统一 top_k={TOP_K}")
        metrics = metrics_for(hits, annotations[case_id]["expected_pages"])
        benchmark[case_id][method] = {
            "metrics": metrics,
            "answer_evidence": answer_evidence(hits),
            "outcome_vs_original": (
                "baseline"
                if method == "原文"
                else outcome_vs_baseline(baseline_metrics, metrics)
            ),
        }

for case_id in QUESTION_IDS:
    print(f"\n问题：{query_by_id[case_id]}")
    for method in METHODS:
        item = benchmark[case_id][method]
        metrics = item["metrics"]
        print(
            f"  {method}：页={metrics['pages']}；必要页首条排名="
            f"{metrics['first_required_rank'] or '未命中'}；覆盖率={metrics['required_page_coverage']:.2f}；"
            f"相对原文={item['outcome_vs_original']}"
        )

# 证明增强字段命中后只映回原文；所有方法共享相同的检索字段字符预算，回答映回只核查来源，不组装/比较回答上下文。
mapping_checks = {}
for method in METHODS:
    mapped = benchmark[QUESTION_IDS[0]][method]["answer_evidence"]
    valid = all(
        item["answer_text"] == row_by_id[item["chunk_id"]]["original_text"]
        and item["answer_text"] == row_by_id[item["chunk_id"]]["answer_text"]
        for item in mapped
    )
    mapping_checks[method] = valid
    print(
        f"映回检查（{method}）：", valid,
        "；首条 answer_text 前 80 字：", mapped[0]["answer_text"][:80],
    )
    if not valid:
        raise AssertionError(f"{method} 的命中结果没有稳定映回原文")

outcome_counts = {
    method: {
        outcome: sum(
            benchmark[case_id][method]["outcome_vs_original"] == outcome
            for case_id in QUESTION_IDS
        )
        for outcome in ("improved", "unchanged", "degraded", "tradeoff")
    }
    for method in METHODS
}
print("\n相对原文的汇总（保留改善、不变和退化状态）：")
for method, counts in outcome_counts.items():
    print("  ", method, counts)


def audit_hit(hit):
    return {
        "chunk_id": hit.chunk_id,
        "pages": list(hit.pages),
        "score": round(float(hit.score), 5),
    }


method_results_for_audit = {}
for case_id in QUESTION_IDS:
    method_results_for_audit[case_id] = {}
    for method in METHODS:
        item = benchmark[case_id][method]
        method_results_for_audit[case_id][method] = {
            "metrics": item["metrics"],
            "outcome_vs_original": item["outcome_vs_original"],
            "hits": [
                audit_hit(hit) for hit in retrieval_results[case_id][method]
            ],
            "answer_chunk_ids": [
                evidence["chunk_id"] for evidence in item["answer_evidence"]
            ],
        }

main_case = QUESTION_IDS[0]
baseline_main = benchmark[main_case]["原文"]["metrics"]
questions_main = benchmark[main_case]["GLM生成问题（Document Augmentation）"]["metrics"]
audit_payload = {
    "case_id": "c3_index_enhancement_comparison",
    "method": "C3 索引增强横向比较",
    "role": "main",
    # 保留标准 before/after 结构，同时在 experiment 中保存全量横向结果。
    "before": baseline_main,
    "after": questions_main,
    "comparison": {
        "name": "GLM 生成问题相对原文的首条必要页排名",
        "before": rank_value(baseline_main["first_required_rank"]),
        "after": rank_value(questions_main["first_required_rank"]),
        "higher_is_better": False,
    },
    "experiment": {
        "question_ids": QUESTION_IDS,
        "queries": query_by_id,
        "candidate_pages": list(CANDIDATE_PAGES),
        "candidate_chunk_ids": [row["chunk_id"] for row in candidate_rows],
        "top_k": TOP_K,
        "search_char_budget": SEARCH_CHAR_BUDGET,
        "shared_search_field_char_budget": SEARCH_CHAR_BUDGET,
        "answer_mapping_rule": "命中后只按稳定 chunk_id 核查并映回原文来源，不组装或比较回答上下文",
        "methods": METHODS,
        "method_results": method_results_for_audit,
        "outcome_counts": outcome_counts,
        "answer_mapping_checks": mapping_checks,
    },
    "generated_fields": {
        chunk_id: {
            "chapter_title": row_by_id[chunk_id]["chapter_title"],
            "generated_questions": row_by_id[chunk_id]["generated_questions"],
            "contextual_background": row_by_id[chunk_id]["contextual_background"],
            "original_text": row_by_id[chunk_id]["original_text"],
            "answer_text": row_by_id[chunk_id]["answer_text"],
            "search_text_by_method": row_by_id[chunk_id]["search_text_by_method"],
            "validation_status": row_by_id[chunk_id]["validation_status"],
            "validation_notes": row_by_id[chunk_id]["validation_notes"],
        }
        for chunk_id in row_by_id
    },
    "generation": {
        "call_count": len(candidate_rows),
        "failure_count": len(generation_failures),
        "raw_model_outputs": {
            chunk_id: record["raw_model_output"]
            for chunk_id, record in generation_records.items()
        },
        "validated_payloads": {
            chunk_id: record.get("parsed_payload")
            for chunk_id, record in generation_records.items()
            if record.get("status") == "validated"
        },
        "validation_status": {
            chunk_id: {
                "status": record["status"],
                "notes": record["validation_notes"],
            }
            for chunk_id, record in generation_records.items()
        },
    },
    "provenance": {
        "status": "fresh_api_run",
        "model": "glm-4-flash",
        "env_file": str(PROJECT_ENV_RELATIVE_PATH),
        "candidate_scope": "fixed 9-page pumpkin-book corpus shared by every method",
        "annotation_usage": "expected_pages loaded only after all retrieval methods finished",
        "failure_policy": "record raw output and validation error, then raise; no fallback or fabricated result",
        "answer_field_rule": "every hit is mapped by stable chunk_id to original_text/answer_text",
    },
}

sample_chunk_id = candidate_rows[0]["chunk_id"]
print("\n原始 GLM 输出样例（完整 raw_model_outputs 已写入审计输出）：")
print(generation_records[sample_chunk_id]["raw_model_output"])
emit_tutorial_audit(audit_payload)


问题：《南瓜书》绪论里说，机器学习算法之间有没有绝对更好的一个？
  原文：页=[2, 14, 17]；必要页首条排名=3；覆盖率=1.00；相对原文=baseline
  可靠章节标题（CCH）：页=[2, 17, 14]；必要页首条排名=2；覆盖率=1.00；相对原文=improved
  GLM生成问题（Document Augmentation）：页=[2, 14, 17]；必要页首条排名=3；覆盖率=1.00；相对原文=unchanged
  GLM背景（Contextual Retrieval）：页=[2, 17, 14]；必要页首条排名=2；覆盖率=1.00；相对原文=improved

问题：南瓜书第 2.2 节介绍了哪三种模型评估方法？
  原文：页=[18, 14, 2]；必要页首条排名=1；覆盖率=1.00；相对原文=baseline
  可靠章节标题（CCH）：页=[18, 14, 2]；必要页首条排名=1；覆盖率=1.00；相对原文=unchanged
  GLM生成问题（Document Augmentation）：页=[18, 2, 14]；必要页首条排名=1；覆盖率=1.00；相对原文=unchanged
  GLM背景（Contextual Retrieval）：页=[18, 19, 14]；必要页首条排名=1；覆盖率=1.00；相对原文=unchanged

问题：集成多个弱学习器来提升整体预测效果的方法是什么
  原文：页=[19, 88, 16]；必要页首条排名=2；覆盖率=1.00；相对原文=baseline
  可靠章节标题（CCH）：页=[19, 88, 18]；必要页首条排名=2；覆盖率=1.00；相对原文=unchanged
  GLM生成问题（Document Augmentation）：页=[88, 16, 19]；必要页首条排名=1；覆盖率=1.00；相对原文=improved
  GLM背景（Contextual Retrieval）：页=[19, 18, 88]；必要页首条排名=3；覆盖率=1.00；相对原文=degraded
映回检查（原文）： True ；首条 answer_text 前 80 字： 前言 “周志华老师的《机器学习》（西瓜书）是机器学习领域的经典入门教材之一，周老师为了使尽可能多的读 

## 如何解读

先看统一配置和调用数，再看每道题四种方法的 `pages`、必要页首条排名、覆盖率及相对原文状态。`degraded` 或 `unchanged` 都是有效实验结果，不应删掉；候选集只有 9 页、问题只有 3 道，不能把它概括成全书收益。

真正接入回答链时，只能把 `answer_text`（原文）组成上下文；`chapter_title`、`generated_questions` 和 `contextual_background` 是索引字段，不能当作证据。若想学习其中某一机制的原理、提示词或失败边界，请回到本节开头列出的三个专题 Notebook。